# Pure Dense Retrieval Pipeline

Runs the selected OCR corpus through article preparation, recursive chunking, BGE-M3 embedding, and ChromaDB indexing. This notebook does not use BM25, an inverted index, or FAISS.

In [1]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'data-pipeline').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'data-pipeline'))

from apps.core.retrieval_config import PipelineConfig
from experiments.pipeline.corpus_selection import select_and_reconcile_ocr

config = PipelineConfig(project_root=PROJECT_ROOT)
config.create_output_dirs()
print('OCR directory:', config.ocr_dir)
print('Documents CSV:', config.documents_csv)
print('Output directory:', config.output_dir)

# Reconcile OCR files first. Files outside the selected domains, and OCR
# files overlapping a core-law title, are moved to data/archive/text-ocr.
ocr_paths, movement = select_and_reconcile_ocr(PROJECT_ROOT, config.documents_csv)
print('OCR reconciliation:', movement)
print('Selected OCR files:', len(ocr_paths))

OCR directory: /Users/nak/CADT/projects/legal-rag/data/text-ocr
Documents CSV: /Users/nak/CADT/projects/legal-rag/data/documents.csv
Output directory: /Users/nak/CADT/projects/legal-rag/data/dense_retrieval


OCR reconciliation: {'active_archived': 0, 'archive_restored': 0, 'selected': 55, 'core_duplicates': 0}
Selected OCR files: 55
Core-law articles: 3337


## Add core legal articles

Load the five structured core laws and append them to the selected OCR articles before chunking and embedding.

In [ ]:
from experiments.pipeline.corpus_selection import load_core_articles

core_articles = load_core_articles(PROJECT_ROOT)
core_ids = [str(article['article_id']) for article in core_articles]
if len(core_ids) != len(set(core_ids)):
    raise ValueError('Duplicate core-law article IDs detected')

print('Core-law articles:', len(core_articles))
print('Core-law titles:', sorted({article['title'] for article in core_articles}))

In [2]:
from dense_retrieval.text_preprocess import prepare_articles

articles = prepare_articles(config, ocr_paths=ocr_paths, extra_articles=core_articles)
print('Prepared articles:', len(articles))
print('Saved:', config.article_path)

Prepared articles: 5111
Saved: /Users/nak/CADT/projects/legal-rag/data/dense_retrieval/articles.jsonl


In [3]:
import importlib
import dense_retrieval.text_preprocess as text_preprocess_module
importlib.reload(text_preprocess_module)
build_chunks = text_preprocess_module.build_chunks

chunks = build_chunks(articles, config)
print('Created chunks:', len(chunks))
print('Saved:', config.chunk_path)

Created chunks: 5476
Saved: /Users/nak/CADT/projects/legal-rag/data/dense_retrieval/chunks.jsonl


In [4]:
from dense_retrieval.embeddings import embed_chunks

vectors = embed_chunks(chunks, config)
print('Embedding shape:', vectors.shape)
print('Saved:', config.embedding_path)

W1109 13:29:47.935000 72515 .venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/2738 [00:00<?, ?it/s]

Embedding shape: (5476, 1024)
Saved: /Users/nak/CADT/projects/legal-rag/data/dense_retrieval/embeddings.npy


In [5]:
from dense_retrieval.chroma_index import build_index

index_info = build_index(chunks, vectors, config)
print(json.dumps(index_info, ensure_ascii=False, indent=2))

{
  "collection": "legal_dense_bge_m3_450",
  "path": "/Users/nak/CADT/projects/legal-rag/data/dense_retrieval/chroma_db",
  "stored_vectors": 5476,
  "dimension": 1024,
  "metric": "cosine"
}


In [6]:
print('Pipeline complete.')
print('Articles:', len(articles))
print('Chunks:', len(chunks))
print('Vectors:', vectors.shape)
print('Collection:', index_info['collection'])

Pipeline complete.
Articles: 5111
Chunks: 5476
Vectors: (5476, 1024)
Collection: legal_dense_bge_m3_450
